In [1]:
!git clone https://github.com/Ukhadija/novel-RAG.git
%cd /kaggle/working/novel-RAG

Cloning into 'novel-RAG'...
remote: Enumerating objects: 76, done.
remote: Counting objects: 100% (76/76), done.
remote: Compressing objects: 100% (52/52), done.
remote: Total 76 (delta 32), reused 66 (delta 22), pack-reused 0 (from 0)
Receiving objects: 100% (76/76), 16.88 KiB | 4.22 MiB/s, done.
Resolving deltas: 100% (32/32), done.
/kaggle/working/novel-RAG


# imports

In [2]:
!git pull

Already up to date.


In [3]:
!pip install -q pdfplumber rank-bm25 tiktoken # sentence-transformers torchvision
!pip install -q accelerate bitsandbytes

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.0/71.0 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 96.9 MB/s eta 0:00:00ta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 105.7 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 106.5 MB/s eta 0:00:0000:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ydata-profiling 4.18.4 requires numba<0.63,>=0.60, but you have numba 0.65.1 which is incompatible.
ydata-profiling 4.18.4 requires numpy<2.4,>=1.22, but you have numpy 2.4.6 which is incompatible.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
gradio 5.50.0 requires pillow<12.0,>=8.0, but you

In [4]:
#!pip install -q --upgrade pillow
!pip uninstall -y torchvision

Found existing installation: torchvision 0.25.0+cu128
Uninstalling torchvision-0.25.0+cu128:
  Successfully uninstalled torchvision-0.25.0+cu128


Extract text from pdf to json

In [5]:
import numpy, torch
print("numpy:", numpy.__version__)
print("torch:", torch.__version__)
from sentence_transformers import SentenceTransformer, CrossEncoder
print("sentence-transformers imported OK")

numpy: 2.4.6
torch: 2.10.0+cu128
sentence-transformers imported OK


In [6]:
import sys
sys.path.insert(0, "/kaggle/working/novel-RAG/src")

from chunking import chunk_pages, extract_corpus
from retrieval import Retriever
from answer_generator import build_prompt_fewshot, generate_answer_v2
from verifier import run_verifier, answer_with_verification
from evaluate import evaluate_single, retrieval_precision_at_k

book_specs = [
    {"path": "/kaggle/input/datasets/umekhadij/novel-dataset/Anne_of_Green_Gables.pdf", "title": "anne_of_green_gable"},
    {"path": "/kaggle/input/datasets/umekhadij/novel-dataset/rilla_of_ingleside.pdf", "title": "rilla_of_ingleside"},
]

# chunk the corpus and process

In [7]:


corpus = extract_corpus(book_specs, out_path="../pages.jsonl")
print(f"\nTotal pages across corpus: {len(corpus)}")


import json

pages = []
with open("../pages.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        pages.append(json.loads(line))

chunks = chunk_pages(pages, target_tokens=600, overlap_tokens=120)
print(f"Produced {len(chunks)} chunks from {len(pages)} pages")
print("\nSample chunk:")
print(json.dumps(chunks[0], indent=2)[:500])

with open("../chunks.jsonl", "w", encoding="utf-8") as f:
    for c in chunks:
        f.write(json.dumps(c, ensure_ascii=False) + "\n")
print(f"\nWrote chunks to /chunks.jsonl")


chunks = []
with open("../chunks.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        chunks.append(json.loads(line))

retriever = Retriever(chunks)


Extracting: anne_of_green_gable (/kaggle/input/datasets/umekhadij/novel-dataset/Anne_of_Green_Gables.pdf)
  -> 316 non-empty pages extracted
Extracting: rilla_of_ingleside (/kaggle/input/datasets/umekhadij/novel-dataset/rilla_of_ingleside.pdf)
  -> 178 non-empty pages extracted
Wrote 494 page records to ../pages.jsonl

Total pages across corpus: 494
Produced 559 chunks from 494 pages

Sample chunk:
{
  "chunk_id": "anne_of_green_gable_00000",
  "book": "anne_of_green_gable",
  "page_start": 3,
  "page_end": 10,
  "text": "The Project Gutenberg eBook of Anne of Green Gables\n\nThis ebook is for the use of anyone anywhere in the United States and most other parts of the world at no cost and with almost no restrictions whatsoever.\n\nYou may copy it, give it away or re-use it under the terms of the Project Gutenberg License included with this ebook or online at www.gutenberg.org.\n\nIf you ar

Wrote chunks to /chunks.jsonl
Building BM25 index over 559 chunks...
Loading embedding model: se

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Retriever ready.



Now Generator model

# model loadded here

In [8]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

# API endpoint

In [9]:
!pip install -q fastapi uvicorn nest-asyncio pyngrok

In [10]:
import importlib
import app as rag_app
importlib.reload(rag_app)
rag_app.retriever = retriever
rag_app.model = model
rag_app.tokenizer = tokenizer
rag_app.generate_answer_v2 = generate_answer_v2
rag_app.answer_with_verification = answer_with_verification
rag_app.evaluate_single_fn = evaluate_single
print("Injected:", hasattr(rag_app, 'generate_answer_v2'))

Injected: True


In [11]:
import nest_asyncio
import uvicorn
from pyngrok import ngrok
import threading

nest_asyncio.apply()
ngrok.set_auth_token("3Fu2rC8JCA2yKeeewqRPQcaknER_3HsB2WvYSP4cvQv13LXFp")
public_url = ngrok.connect(8000)
print(f"Public API URL: {public_url}")
print(f"Interactive docs: {public_url}/docs")

# Run uvicorn in a background thread so the Colab cell doesn't block
config = uvicorn.Config(rag_app.app, host="0.0.0.0", port=8000)
server = uvicorn.Server(config)

def run():
    server.run()

thread = threading.Thread(target=run, daemon=True)
thread.start()

Public API URL: NgrokTunnel: "https://cramp-jasmine-regular.ngrok-free.dev" -> "http://localhost:8000"
Interactive docs: NgrokTunnel: "https://cramp-jasmine-regular.ngrok-free.dev" -> "http://localhost:8000"/docs


In [12]:
#server.should_exit = True

## TEST query

In [14]:
import requests

base = "http://localhost:8000"

# Health check
resp = requests.get(f"{base}/health")
print("Health:", resp.json())

# Query
resp = requests.post(f"{base}/query", json={
    "question": "Does Catherina like Anne?",
    "top_k": 5,
    "use_verification": True,
})
print("\nQuery response:")
import json
print(json.dumps(resp.json(), indent=2))



INFO:     127.0.0.1:37324 - "GET /health HTTP/1.1" 200 OK
Health: {'status': 'ok', 'model_loaded': True, 'chunks_loaded': 559}


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


INFO:     127.0.0.1:37330 - "POST /query HTTP/1.1" 200 OK

Query response:
{
  "question": "Does Catherina like Anne?",
  "answer": "I don't have enough information in the provided context to answer this.",
  "final_verdict": "APPROVE",
  "retrieved_chunks": [
    {
      "chunk_id": "anne_of_green_gable_00159",
      "book": "anne_of_green_gable",
      "page_start": 185,
      "page_end": 186,
      "rerank_score": -1.5201
    },
    {
      "chunk_id": "anne_of_green_gable_00023",
      "book": "anne_of_green_gable",
      "page_start": 32,
      "page_end": 34,
      "rerank_score": -1.8069
    },
    {
      "chunk_id": "anne_of_green_gable_00222",
      "book": "anne_of_green_gable",
      "page_start": 251,
      "page_end": 253,
      "rerank_score": -2.0821
    },
    {
      "chunk_id": "anne_of_green_gable_00176",
      "book": "anne_of_green_gable",
      "page_start": 204,
      "page_end": 205,
      "rerank_score": -2.9393
    },
    {
      "chunk_id": "anne_of_green_ga

In [15]:


import requests
import json

base = "http://localhost:8000"

resp = requests.post(f"{base}/eval", json={
    "id": "test_001",
    "category": "character_relationships",
    "query": "Who is Anne Shirley's best friend?",
    "expected_answer": "Diana Barry",
})

print(json.dumps(resp.json(), indent=2))

INFO:     127.0.0.1:36256 - "POST /eval HTTP/1.1" 200 OK
{
  "id": "test_001",
  "category": "character_relationships",
  "query": "Who is Anne Shirley's best friend?",
  "final_answer": "I don't have enough verified information in the source material to answer this confidently.",
  "final_verdict": "REJECT",
  "latency_seconds": 24.59724497795105,
  "retrieved_chunk_ids": [
    "anne_of_green_gable_00252",
    "anne_of_green_gable_00164",
    "anne_of_green_gable_00218",
    "anne_of_green_gable_00075",
    "anne_of_green_gable_00137"
  ],
  "is_refusal": true
}
